# AlgoPerf: Vanilla vs. Sharded Muon Implementation Study


## 1. Imports & Repo Root

Locates the repo root (so the notebook works whether launched from the root or
from this folder), then imports the scoring package.


In [ ]:
import os
import pickle
import sys
from pathlib import Path

# Run everything relative to the repo root so `scoring` imports and the
# repo-relative paths in Section 3 work from any launch directory.
REPO_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'scoring' / 'score_submissions.py').exists()
)
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from tabulate import tabulate

from scoring import performance_profile, scoring_utils
from scoring.config import DEFAULT_TARGETS_PATH, WorkloadConfig


## 2. Configuration

Set the paths and flags below before running the rest of the notebook.

In [ ]:
# ── Required ──────────────────────────────────────────────────────────────────
# Path to the directory that contains one sub-folder per submission
# (relative to the repo root).
SUBMISSION_DIRECTORY = 'logs/self_tuning'

# Where to write output CSVs, plots, and LaTeX tables. This is the committed
# artifact directory for the second scoring iteration.
OUTPUT_DIR = 'artifacts/tech_report_v1/section_6_algorithms'

# ── Submission filters (leave empty strings to include/exclude nothing) ───────
# Comma-separated names to include (empty = include all).
INCLUDE_SUBMISSIONS = ''
# Comma-separated names to exclude.
EXCLUDE_SUBMISSIONS = 'muon_torch_jax_hps,muon_torch_jax_hps_achandr,muon_torch_jax_hps_lr_fix,muon_torch_replicated_jax_hps,muon_torch_replicated_torch_hps'

# ── Scoring flags ─────────────────────────────────────────────────────────────
# Set True to enforce the competition's strict trial/study count rules.
STRICT = False
# Set True when scoring the self-tuning ruleset.
SELF_TUNING_RULESET = True
# Set True to compute and plot performance profiles after building summaries.
COMPUTE_PERFORMANCE_PROFILES = True
# Benchmark version config: base/held-out workloads, targets, step hints.
# The score divides by the number of base workloads in this config.
WORKLOAD_CONFIG = WorkloadConfig.from_json(DEFAULT_TARGETS_PATH)

# ── Performance profile parameters ────────────────────────────────────────────
MIN_TAU = 1.0
MAX_TAU = 4.0   # set None to auto-detect from data
NUM_POINTS = 100
SCALE = 'linear'  # 'linear' or 'log'

# ── Caching (optional) ────────────────────────────────────────────────────────
# Save the parsed results dict so you can reload it later without re-parsing.
SAVE_RESULTS_TO = None   # e.g. 'results.pkl'
# Load a previously saved results dict instead of re-parsing.
LOAD_RESULTS_FROM = None  # e.g. 'results.pkl'

os.makedirs(OUTPUT_DIR, exist_ok=True)


## 3. Implementation Study: Vanilla vs. Sharded Muon

The four PyTorch Muon submissions form two matched pairs that run the same
update rule with the same batch sizes under two fixed hyperparameter configs,
differing only in implementation: sharded (`MuonDataParallel`, distributes the
Newton-Schulz orthogonalization across GPUs with overlapped collectives) vs.
vanilla (`MuonSingleDevice`, every GPU orthogonalizes every layer). Together
with the native JAX submission (which uses the same HPs as the JAX-HP pair)
they also form a ladder in which each step changes one factor: framework,
sharding, or the fixed hyperparameters. These submissions are excluded from
the leaderboard pool above, so their logs are loaded here separately. Emits
`muon_implementation_table.tex` (within-workload vanilla / sharded ratios),
`muon_waterfall_table.tex` (score ladder), and `muon_implementation.csv`
(raw medians).


In [ ]:
# ── Implementation study: vanilla (replicated) vs sharded Muon ───────────────
# Four PyTorch Muon submissions implement the same update rule with the same
# batch sizes: a sharded implementation (MuonDataParallel: reduce-scatter
# gradients, orthogonalize 1/world_size of the layers per GPU, all-gather) and
# a vanilla single-device implementation (MuonSingleDevice: every GPU
# orthogonalizes every layer), each under two fixed hyperparameter configs.
# These pairs isolate implementation quality from the algorithm; together with
# the native JAX submission (which uses the same HPs as the JAX-HP pair) they
# form a ladder in which each step changes one factor.
# Emits muon_implementation_table.tex, muon_waterfall_table.tex, and
# muon_implementation.csv.

MUON_PAIRS = {  # hp label -> (sharded folder, vanilla folder)
    'Torch HPs': ('muon_torch', 'muon_torch_replicated_torch_hps'),
    'JAX HPs': ('muon_torch_jax_hps_lr_fix', 'muon_torch_replicated_jax_hps'),
}
# Benchmark scores from the full-pool scoring run committed on main
# (results/scores.csv). Adding the extra Muon variants to the pool leaves all
# other submissions' scores unchanged (verified: identical to
# artifacts/leaderboard_v2/scores.csv on every common submission).
MUON_SCORES = {
    'muon': 0.2845,
    'muon_torch': 0.4231,
    'muon_torch_replicated_torch_hps': 0.3827,
    'muon_torch_jax_hps_lr_fix': 0.3137,
    'muon_torch_replicated_jax_hps': 0.2969,
}

MUON_WLS = ['criteo1tb', 'fastmri', 'finewebedu_lm', 'imagenet_resnet',
            'imagenet_vit', 'librispeech_conformer', 'librispeech_deepspeech',
            'ogbg', 'wmt']

muon_folders = sorted({f for pair in MUON_PAIRS.values() for f in pair})
muon_results = {}
for sub in muon_folders:
    muon_results[sub] = scoring_utils.get_experiment_df(
        os.path.join(SUBMISSION_DIRECTORY, sub))

muon_ttt, muon_stt, muon_st = {}, {}, {}
for sub, df in muon_results.items():
    muon_ttt[sub] = performance_profile.get_workloads_time_to_target(
        df, sub, WORKLOAD_CONFIG, time_col='score', verbosity=0,
        self_tuning_ruleset=True, strict=False).iloc[0]
    muon_stt[sub] = performance_profile.get_workloads_time_to_target(
        df, sub, WORKLOAD_CONFIG, time_col='global_step', verbosity=0,
        self_tuning_ruleset=True, strict=False).iloc[0]
    # Per-trial average step time; same methodology as the step-time table.
    rows = {}
    for workload, group in df.groupby('workload'):
        base = workload.replace('_pytorch', '').replace('_jax', '')
        vals = []
        for _, trial in group.iterrows():
            t = np.asarray(trial['accumulated_submission_time'])
            s = np.asarray(trial['global_step'])
            if len(t) >= 2 and s[-1] > s[0]:
                vals.append((t[-1] - t[0]) / (s[-1] - s[0]))
        rows[base] = vals
    muon_st[sub] = rows

pd.concat({'time_to_target_s': pd.DataFrame(muon_ttt).T,
           'steps_to_target': pd.DataFrame(muon_stt).T,
           'mean_step_time_s': pd.DataFrame(
               {s: {w: np.mean(v) for w, v in r.items()}
                for s, r in muon_st.items()}).T},
          names=['metric', 'submission']).to_csv(
    os.path.join(OUTPUT_DIR, 'muon_implementation.csv'))

# ── LaTeX table: within-workload ratios, vanilla ÷ sharded ────────────────────
_WL_MACRO = {
    'criteo1tb': r'\criteo', 'fastmri': r'\fastmri',
    'finewebedu_lm': r'\finewebedu', 'imagenet_resnet': r'\resnet',
    'imagenet_vit': r'\vit', 'librispeech_conformer': r'\conformer',
    'librispeech_deepspeech': r'\deepspeech', 'ogbg': r'\ogbg', 'wmt': r'\wmt',
}

def _ratio(metric, hp, wl):
    shard, van = MUON_PAIRS[hp]
    s, v = metric[shard][wl], metric[van][wl]
    if np.isfinite(s) and np.isfinite(v):
        return f'${v / s:.2f}\\times$'
    if np.isfinite(s) != np.isfinite(v):  # not hit by current data
        return r'$\infty$' if np.isfinite(s) else r'$0$'
    return r'\textemdash{}'

# Only workloads where at least one variant reaches the target.
tbl_wls = [w for w in MUON_WLS
           if any(np.isfinite(muon_ttt[s][w]) for s in muon_folders)]

lines = [
    r'\begin{table}[htbp]',
    r'  \centering',
    r'  \caption{Vanilla $\div$ sharded ratios of median steps and time to '
    r'target (three studies) for the PyTorch Muon implementations, under the '
    r'two fixed hyperparameter configurations. Above $1\times$: vanilla '
    r'needs more; \textemdash{}: target not reached. The \imagenet and '
    r'\librispeech workloads, whose targets no variant reaches, are omitted. '
    r'Scores from the augmented leaderboard pool (\cref{tab:scores} '
    r'unchanged).}',
    r'  \label{tab:muon_implementation}',
    r'  \begin{tabular}{lrrrr}',
    r'    \toprule',
    r'    & \multicolumn{2}{c}{Torch-tuned HPs} & '
    r'\multicolumn{2}{c}{JAX-tuned HPs} \\',
    r'    \cmidrule(lr){2-3} \cmidrule(lr){4-5}',
    r'    Workload & Steps to target & Time to target '
    r'& Steps to target & Time to target \\',
    r'    \midrule',
]
for wl in tbl_wls:
    row = [_ratio(muon_stt, 'Torch HPs', wl), _ratio(muon_ttt, 'Torch HPs', wl),
           _ratio(muon_stt, 'JAX HPs', wl), _ratio(muon_ttt, 'JAX HPs', wl)]
    lines.append('    ' + _WL_MACRO[wl] + ' & ' + ' & '.join(row) + r' \\')
lines += [
    r'    \midrule',
    r'    Benchmark score & \multicolumn{2}{c}{$0.4231 \rightarrow 0.3827$} & '
    r'\multicolumn{2}{c}{$0.3137 \rightarrow 0.2969$} \\',
    r'    \bottomrule',
    r'  \end{tabular}',
    r'\end{table}',
]
muon_table_path = os.path.join(OUTPUT_DIR, 'muon_implementation_table.tex')
with open(muon_table_path, 'w') as f:
    f.write('\n'.join(lines) + '\n')
print(f'LaTeX saved -> {muon_table_path}')
print('\n'.join(lines))

# ── LaTeX table: score waterfall across the Muon family ───────────────────────
# All five Muon submissions as a ladder in which every row changes exactly
# one factor (bolded cell) relative to the previous row. The JAX submission
# keeps params and optimizer state replicated (submission.py in_shardings),
# so its orthogonalization is organized like the vanilla PyTorch
# implementation and the first rung isolates the framework port. The final
# row steps back down from muon_torch to the replicated implementation under
# Torch HPs, exposing the path dependence of the implementation rung.
LADDER = [  # (folder, framework cell, implementation cell, hp cell)
    ('muon', 'JAX', 'vanilla', 'JAX'),
    ('muon_torch_replicated_jax_hps', r'\textbf{PyTorch}', 'vanilla', 'JAX'),
    ('muon_torch_jax_hps_lr_fix', 'PyTorch', r'\textbf{sharded}', 'JAX'),
    ('muon_torch', 'PyTorch', 'sharded', r'\textbf{Torch}'),
    ('muon_torch_replicated_torch_hps', 'PyTorch', r'\textbf{vanilla}',
     'Torch'),
]
wf_lines = [
    r'\begin{table}[htbp]',
    r'  \centering',
    r'  \caption{The five Muon submissions; each row changes one factor '
    r'(bold) relative to the row above. Rows one and four are the '
    r'leaderboard entries \muonjax{} and \muonpt{}; the final row steps '
    r'back down to the vanilla implementation.}',
    r'  \label{tab:muon_waterfall}',
    r'  \begin{tabular}{lllrr}',
    r'    \toprule',
    r'    Framework & Implementation & HPs & Score & $\Delta$ \\',
    r'    \midrule',
]
prev = None
for sub, fw, orth, hp in LADDER:
    score = MUON_SCORES[sub]
    delta = '---' if prev is None else f'${score - prev:+.3f}$'
    wf_lines.append(f'    {fw} & {orth} & {hp} & ${score:.4f}$ & {delta} \\\\')
    prev = score
wf_lines += [r'    \bottomrule', r'  \end{tabular}', r'\end{table}']
waterfall_path = os.path.join(OUTPUT_DIR, 'muon_waterfall_table.tex')
with open(waterfall_path, 'w') as f:
    f.write('\n'.join(wf_lines) + '\n')
print(f'LaTeX saved -> {waterfall_path}')
print('\n'.join(wf_lines))
